In [1]:
from pathlib import Path
import pandas as pd
import shutil
import textwrap

BASE_DIR = Path(r"D:\ai_supply_chain_risk")
OUTPUT_DIR = BASE_DIR / "outputs"
NOTE06_FILE = OUTPUT_DIR / "note06_final_business_insights.xlsx"

PORTFOLIO_DIR = OUTPUT_DIR / "portfolio_assets"
PORTFOLIO_DIR.mkdir(parents=True, exist_ok=True)

GITHUB_ASSETS_DIR = BASE_DIR / "github_assets"
GITHUB_ASSETS_DIR.mkdir(parents=True, exist_ok=True)

CHART_DIR = OUTPUT_DIR / "note06_charts"

print("Note06 file exists:", NOTE06_FILE.exists())
print("Portfolio dir:", PORTFOLIO_DIR)
print("GitHub assets dir:", GITHUB_ASSETS_DIR)

Note06 file exists: True
Portfolio dir: D:\ai_supply_chain_risk\outputs\portfolio_assets
GitHub assets dir: D:\ai_supply_chain_risk\github_assets


In [2]:
latest_ranking = pd.read_excel(NOTE06_FILE, sheet_name="latest_year_ranking", dtype={"stock_code_fixed": str})
trend_summary = pd.read_excel(NOTE06_FILE, sheet_name="company_risk_trend", dtype={"stock_code_fixed": str})
watchlist_final = pd.read_excel(NOTE06_FILE, sheet_name="final_watchlist", dtype={"stock_code_fixed": str})

try:
    cluster_change_summary = pd.read_excel(NOTE06_FILE, sheet_name="cluster_change_summary")
except:
    cluster_change_summary = pd.DataFrame()

try:
    cluster_change_detail = pd.read_excel(NOTE06_FILE, sheet_name="cluster_change_detail", dtype={"stock_code_fixed": str})
except:
    cluster_change_detail = pd.DataFrame()

print("latest_ranking:", latest_ranking.shape)
print("trend_summary:", trend_summary.shape)
print("watchlist_final:", watchlist_final.shape)
print("cluster_change_summary:", cluster_change_summary.shape)

display(watchlist_final)

latest_ranking: (10, 6)
trend_summary: (10, 12)
watchlist_final: (4, 20)
cluster_change_summary: (2, 2)


,stock_code_fixed,company_name,first_year,latest_year,first_risk_score,latest_risk_score,risk_change,risk_change_pct,risk_trend_slope,risk_trend_label,observations,latest_cluster_profile_label,is_latest_high_risk,is_risk_increasing,is_large_risk_increase,cluster_requires_attention,has_core_risk_signal,watchlist_score,watchlist_tier,watchlist_reason
0,000034,神州数码,2021,2025,50.85,64.05,13.20,0.259587,3.465,risk_increasing,5,high_exposure_profile,True,True,True,True,True,4,main_watchlist,最新年度风险得分位于样本前25%；风险得分呈明显上升趋势；相较基准年份风险增幅较大；聚类画像...
1,000977,浪潮信息,2021,2025,45.55,59.00,13.45,0.295280,3.310,risk_increasing,5,high_exposure_profile,True,True,True,True,True,4,main_watchlist,最新年度风险得分位于样本前25%；风险得分呈明显上升趋势；相较基准年份风险增幅较大；聚类画像...
2,601138,工业富联,2021,2025,60.70,62.90,2.20,0.036244,0.490,risk_stable,5,high_exposure_profile,True,False,False,True,True,2,main_watchlist,最新年度风险得分位于样本前25%；聚类画像显示供应链暴露程度或经营压力需要关注
3,000938,紫光股份,2021,2025,39.65,40.15,0.50,0.012610,0.730,risk_stable,5,high_exposure_profile,False,False,False,True,False,1,supplementary_attention,聚类画像显示供应链暴露程度或经营压力需要关注


In [3]:
latest_year = int(latest_ranking["year"].max())
year_min = int(trend_summary["first_year"].min())
year_max = int(trend_summary["latest_year"].max())
company_count = int(trend_summary["stock_code_fixed"].nunique())

main_watchlist = watchlist_final[watchlist_final["watchlist_tier"] == "main_watchlist"].copy()
supplementary_attention = watchlist_final[watchlist_final["watchlist_tier"] == "supplementary_attention"].copy()

top_company = latest_ranking.iloc[0]["company_name"]
top_score = latest_ranking.iloc[0]["risk_score"]

increasing_count = int((trend_summary["risk_trend_label"] == "risk_increasing").sum())
stable_count = int((trend_summary["risk_trend_label"] == "risk_stable").sum())
decreasing_count = int((trend_summary["risk_trend_label"] == "risk_decreasing").sum())

print("Latest year:", latest_year)
print("Year range:", year_min, "-", year_max)
print("Company count:", company_count)
print("Top company:", top_company, top_score)
print("Main watchlist count:", len(main_watchlist))
print("Supplementary attention count:", len(supplementary_attention))
print("Increasing:", increasing_count)
print("Stable:", stable_count)
print("Decreasing:", decreasing_count)

Latest year: 2025
Year range: 2021 - 2025
Company count: 10
Top company: 神州数码 64.05
Main watchlist count: 3
Supplementary attention count: 1
Increasing: 2
Stable: 5
Decreasing: 3


In [5]:
# 生成 README.md
# 这个版本不用三引号，避免 IncompleteInputError

def make_company_list(df):
    if df.empty:
        return "- None"
    
    lines = []
    for _, row in df.iterrows():
        company = row.get("company_name", "")
        code = row.get("stock_code_fixed", "")
        reason = row.get("watchlist_reason", "")
        lines.append(f"- {company} ({code}): {reason}")
    
    return "\n".join(lines)


main_watchlist_text = make_company_list(main_watchlist)
supplementary_text = make_company_list(supplementary_attention)

readme_lines = [
    "# AI Supply Chain Risk Analysis",
    "",
    "## Project Overview",
    "",
    "This project builds a data-driven supply chain risk screening framework for selected Chinese listed companies in the digital infrastructure and technology manufacturing sectors.",
    "",
    f"The project uses company-level financial and operating indicators from {year_min} to {year_max}, constructs supply-chain-related risk features, generates annual risk scores, applies machine learning validation and clustering analysis, and finally produces a business-oriented company watchlist.",
    "",
    "The goal is not to predict that a specific company will definitely experience supply chain disruption. Instead, the project provides a structured screening framework to identify companies that may require further due diligence.",
    "",
    "---",
    "",
    "## Dataset Scope",
    "",
    f"- Observation period: {year_min}-{year_max}",
    f"- Number of companies: {company_count}",
    "- Main sample coverage: listed companies related to digital infrastructure, servers, electronic manufacturing, PCB, and ICT supply chain",
    "- Data type: company financial and operating indicators manually collected and standardized from public annual reports",
    "",
    "---",
    "",
    "## Project Workflow",
    "",
    "1. Data collection and initial inspection",
    "2. Data cleaning and field standardization",
    "3. Feature engineering for supply chain risk indicators",
    "4. Annual risk scoring and company ranking",
    "5. Machine learning validation and clustering analysis",
    "6. Final business interpretation and watchlist construction",
    "7. Portfolio and reporting asset generation",
    "",
    "---",
    "",
    "## Key Outputs",
    "",
    "### 1. Annual Risk Ranking",
    "",
    f"In the latest year, the highest risk score company is **{top_company}**, with a risk score of **{top_score:.2f}**.",
    "",
    "### 2. Risk Trend Analysis",
    "",
    f"- Risk increasing companies: {increasing_count}",
    f"- Risk stable companies: {stable_count}",
    f"- Risk decreasing companies: {decreasing_count}",
    "",
    "### 3. Final Watchlist",
    "",
    "The final watchlist is not based on a single-year score only. It combines:",
    "",
    "- Latest-year risk level",
    "- Risk score trend",
    "- Risk score increase from the base year",
    "- Cluster profile label as supplementary interpretation",
    "",
    "Main watchlist companies:",
    "",
    main_watchlist_text,
    "",
    "Supplementary attention companies:",
    "",
    supplementary_text,
    "",
    "---",
    "",
    "## Business Interpretation",
    "",
    "The final result suggests that companies such as **神州数码** and **浪潮信息** deserve closer attention because they show both high latest-year risk levels and clear upward risk trends.",
    "",
    "**工业富联** is better interpreted as a comparison case: its latest-year risk level is relatively high, but its risk trend is more stable, so it should not be described as a rapidly deteriorating case.",
    "",
    "**紫光股份**, if included, should only be treated as a supplementary observation object because its risk score did not show clear deterioration.",
    "",
    "---",
    "",
    "## Project Limitations",
    "",
    "This project should be interpreted as a risk screening framework rather than a definitive prediction model.",
    "",
    "Main limitations include:",
    "",
    "1. The sample size is relatively small.",
    "2. Some supply chain variables are proxy indicators due to disclosure limitations in annual reports.",
    "3. The risk score depends on feature design and weighting assumptions.",
    "4. Clustering results are used for profile interpretation, not direct risk classification.",
    "5. The framework identifies companies for further review but does not replace fundamental research or field due diligence.",
    "",
    "---",
    "",
    "## Tools Used",
    "",
    "- Python",
    "- pandas",
    "- numpy",
    "- matplotlib",
    "- scikit-learn",
    "- openpyxl",
    "- Jupyter Notebook / VS Code",
    "",
    "---",
    "",
    "## Repository Structure",
    "",
    "ai_supply_chain_risk/",
    "notebooks/",
    "- Note01_data_loading.ipynb",
    "- Note02_data_cleaning.ipynb",
    "- Note03_feature_engineering.ipynb",
    "- Note04_risk_scoring.ipynb",
    "- Note05_ml_validation.ipynb",
    "- Note06_final_business_insights.ipynb",
    "- Note07_portfolio_report_assets.ipynb",
    "",
    "outputs/",
    "- note06_final_business_insights.xlsx",
    "- note06_final_business_summary.txt",
    "- note06_charts/",
    "",
    "github_assets/",
    "",
    "README.md",
    "requirements.txt",
    ".gitignore",
]

readme_text = "\n".join(readme_lines)

README_FILE = BASE_DIR / "README.md"

with open(README_FILE, "w", encoding="utf-8") as f:
    f.write(readme_text)

print("README saved:")
print(README_FILE)

README saved:
D:\ai_supply_chain_risk\README.md


In [7]:
# Note07 第 5 格：生成简历项目描述

resume_lines = [
    "项目名称：AI 供应链风险分析项目",
    "",
    "项目描述：",
    f"基于 {year_min}-{year_max} 年中国科技制造与数字基础设施相关上市公司的财务及经营数据，构建供应链风险筛查框架。项目完成了数据清洗、特征工程、风险评分、机器学习验证、聚类画像分析与最终 watchlist 输出，用于识别需要进一步尽调的潜在供应链风险公司。",
    "",
    "简历版本 1：",
    "- 基于上市公司年报数据，使用 Python 构建供应链风险分析框架，完成数据清洗、特征工程、风险评分、聚类分析与风险名单输出。",
    "- 设计多维度风险指标体系，覆盖盈利能力、偿债压力、现金流质量、研发投入、海外收入暴露及供应链相关经营变量。",
    "- 结合年度风险排名、风险趋势斜率和 cluster profile label，识别神州数码、浪潮信息等重点关注公司，并形成可解释的业务分析结论。",
    "",
    "简历版本 2：",
    f"- 使用 pandas、numpy、matplotlib 和 scikit-learn，对 {company_count} 家上市公司 {year_min}-{year_max} 年财务与经营数据进行供应链风险分析。",
    "- 构建公司级风险评分和年度排名，并通过聚类方法生成风险画像标签，辅助解释不同公司的风险结构差异。",
    "- 输出 final watchlist、趋势分析表和可视化图表，将技术结果转化为面向风控、投研和供应链分析场景的业务判断。",
    "",
    "简历版本 3：",
    "- 独立完成 AI 供应链风险分析项目，从年报数据标准化、风险指标构建到机器学习验证和最终业务解释，形成完整数据分析闭环。",
    "- 通过风险得分、趋势变化和聚类画像综合筛选重点公司，避免仅依赖单一年份排名造成误判。",
    "- 最终生成 Excel 分析结果、项目图表、README 文档和面试展示材料，可用于作品集展示。",
]

resume_text = "\n".join(resume_lines)

RESUME_FILE = PORTFOLIO_DIR / "resume_project_bullets.txt"

with open(RESUME_FILE, "w", encoding="utf-8") as f:
    f.write(resume_text)

print("Resume bullets saved:")
print(RESUME_FILE)

Resume bullets saved:
D:\ai_supply_chain_risk\outputs\portfolio_assets\resume_project_bullets.txt


In [8]:
interview_text = f"""
1 分钟项目介绍：

我做了一个 AI 供应链风险分析项目，主要基于 {year_min} 到 {year_max} 年中国科技制造和数字基础设施相关上市公司的公开年报数据，构建一个供应链风险筛查框架。

这个项目不是预测某家公司一定会发生风险，而是通过财务和经营指标识别哪些公司值得进一步关注。我先对年报数据进行了清洗和标准化，然后设计了风险相关特征，比如盈利能力、偿债压力、现金流质量、研发投入、海外收入暴露和供应链相关代理变量。之后我构建了年度风险得分和排名，并用机器学习验证和聚类方法辅助解释公司的风险画像。

最后，我没有只看单一年份排名，而是结合最新年度风险水平、风险变化趋势、风险增幅和 cluster profile label，生成了 final watchlist。结果中，神州数码和浪潮信息属于风险水平较高且风险趋势上升的重点案例，工业富联更像是高风险水平但趋势相对稳定的对照案例。

---

3 分钟项目介绍：

这个项目的背景是，很多上市公司年报里不会直接披露非常完整的供应链风险数据，尤其是客户集中度、海外供应链依赖、关键供应商等信息经常不完整。所以我选择用财务和经营指标构建一个供应链风险筛查框架，而不是直接做一个不可解释的预测模型。

第一步是数据整理。我手动收集并标准化了 {company_count} 家中国科技制造和数字基础设施相关上市公司在 {year_min} 到 {year_max} 年的财务与经营数据。为了方便 Python 读取，我统一了字段名称、股票代码格式和金额单位，也处理了前导 0 丢失的问题。

第二步是特征工程。我没有只使用单一财务指标，而是构建了多维度风险特征，包括盈利能力、偿债压力、现金流质量、研发投入强度、海外收入或出口代理变量，以及部分供应链暴露相关指标。这样可以更接近实际业务里的风险分析逻辑。

第三步是风险评分和模型验证。我构建了公司年度风险得分和排名，并用机器学习方法进行验证。这里的机器学习不是为了宣传模型准确率，而是为了检查这些风险指标是否能形成相对稳定的区分能力。同时，我使用聚类方法给公司生成 cluster_profile_label，用于解释不同公司的风险结构画像。

第四步是业务解释。我最终没有直接把某一个 cluster 叫作 high risk cluster，因为这样容易误导。Instead，我把 cluster 作为辅助画像，再结合最新年度风险得分、风险趋势斜率和基准年份以来的风险变化，生成 final watchlist。

最终结果显示，神州数码和浪潮信息同时满足最新年度风险较高、风险趋势明显上升、风险增幅较大等条件，因此适合作为重点关注案例。工业富联虽然最新年度风险得分较高，但趋势相对稳定，因此我把它作为对照案例。紫光股份只因为 cluster profile 显示供应链暴露程度需要关注，所以我把它放在 supplementary attention，而不是主风险名单。

这个项目的核心价值是把公开年报数据转化为一个可解释、可复现的风险筛查流程，适合用于风控、投研、供应链分析或企业经营分析的初步筛选场景。

---

可能被追问的问题：

Q1：你这个项目是不是在预测供应链风险事件？
A：不是。我的项目定位是风险筛查和排序，不是事件预测。因为供应链风险事件本身缺少统一标签，而且上市公司披露有限，所以我更强调可解释的指标体系和 watchlist 输出。

Q2：为什么使用 cluster？
A：cluster 的作用不是直接判断高风险，而是帮助解释公司风险结构。例如有些公司可能风险得分不高，但供应链暴露程度较高；有些公司风险水平高但趋势稳定。cluster 可以帮助补充解释这些差异。

Q3：为什么不把 high_exposure_profile 叫 high_risk_cluster？
A：因为 high exposure 只代表供应链或经营暴露程度高，不等于一定高风险。为了避免误导，我使用 cluster_profile_label，并在最终 watchlist 中把它作为辅助解释因素。

Q4：这个项目最大的局限是什么？
A：主要是样本量较小，部分供应链变量只能用代理指标，且年报披露存在不一致。因此这个项目适合做初步筛查，不能替代完整尽调。

Q5：如果继续优化，你会怎么做？
A：我会扩大样本公司数量，加入更多文本信息，比如年报中的供应链风险描述、客户集中度和海外业务披露，并进一步验证风险评分与后续经营波动之间的关系。
"""

interview_text = textwrap.dedent(interview_text).strip()

INTERVIEW_FILE = PORTFOLIO_DIR / "interview_talking_points.txt"
with open(INTERVIEW_FILE, "w", encoding="utf-8") as f:
    f.write(interview_text)

print("Interview talking points saved:")
print(INTERVIEW_FILE)

Interview talking points saved:
D:\ai_supply_chain_risk\outputs\portfolio_assets\interview_talking_points.txt


In [9]:
limitations_text = """
Project Limitations

1. Sample size limitation
The current sample includes a limited number of listed companies, so the result should be interpreted as a demonstration of analytical framework rather than a statistically comprehensive industry conclusion.

2. Proxy variable limitation
Some supply chain risk factors are not directly disclosed in annual reports. Therefore, several indicators are constructed as proxy variables, such as overseas revenue exposure, export sales proxy, cash flow pressure, and operating efficiency indicators.

3. No direct event label
This project does not use a direct supply chain disruption event label. The risk score is designed for screening and ranking, not for predicting a confirmed future event.

4. Cluster interpretation limitation
Cluster labels are used to describe company risk profiles. A cluster profile should not be directly interpreted as a confirmed high-risk classification.

5. Disclosure inconsistency
Different companies disclose financial and operating details in different formats, which may affect comparability across companies and years.

6. Practical usage
The output should be used as a starting point for further due diligence, not as a standalone investment or credit decision.
"""

limitations_text = textwrap.dedent(limitations_text).strip()

LIMITATIONS_FILE = PORTFOLIO_DIR / "project_limitations.txt"
with open(LIMITATIONS_FILE, "w", encoding="utf-8") as f:
    f.write(limitations_text)

print("Limitations file saved:")
print(LIMITATIONS_FILE)

Limitations file saved:
D:\ai_supply_chain_risk\outputs\portfolio_assets\project_limitations.txt


In [10]:
if CHART_DIR.exists():
    chart_files = list(CHART_DIR.glob("*.png"))
    print("Found chart files:", len(chart_files))
    
    for file in chart_files:
        target = GITHUB_ASSETS_DIR / file.name
        shutil.copy2(file, target)
        print("Copied:", target)
else:
    print("Chart dir not found:", CHART_DIR)

Found chart files: 3
Copied: D:\ai_supply_chain_risk\github_assets\note06_latest_year_cluster_distribution.png
Copied: D:\ai_supply_chain_risk\github_assets\note06_latest_year_top_10_risk_companies.png
Copied: D:\ai_supply_chain_risk\github_assets\note06_top_15_risk_increase_companies.png


In [11]:
requirements_text = """
pandas
numpy
matplotlib
scikit-learn
openpyxl
jupyter
"""

gitignore_text = """
.ipynb_checkpoints/
__pycache__/
*.pyc
.env
.venv/
.DS_Store

# Optional: ignore large or raw data files if needed
data/raw/
outputs/*.xlsx
outputs/*.csv
"""

REQUIREMENTS_FILE = BASE_DIR / "requirements.txt"
GITIGNORE_FILE = BASE_DIR / ".gitignore"

with open(REQUIREMENTS_FILE, "w", encoding="utf-8") as f:
    f.write(textwrap.dedent(requirements_text).strip())

with open(GITIGNORE_FILE, "w", encoding="utf-8") as f:
    f.write(textwrap.dedent(gitignore_text).strip())

print("requirements.txt saved:", REQUIREMENTS_FILE)
print(".gitignore saved:", GITIGNORE_FILE)

requirements.txt saved: D:\ai_supply_chain_risk\requirements.txt
.gitignore saved: D:\ai_supply_chain_risk\.gitignore


In [12]:
checklist_text = f"""
Final Project Checklist

Core notebooks:
[ ] Note01 completed
[ ] Note02 completed
[ ] Note03 completed
[ ] Note04 completed
[ ] Note05 completed
[ ] Note06 completed
[ ] Note07 completed

Core outputs:
[ ] note06_final_business_insights.xlsx exists
[ ] final_watchlist sheet exists
[ ] note06 charts exist
[ ] README.md exists
[ ] requirements.txt exists
[ ] .gitignore exists
[ ] resume_project_bullets.txt exists
[ ] interview_talking_points.txt exists

Business conclusion:
[ ] Do not describe the project as predicting actual supply chain disruptions
[ ] Describe it as a risk screening framework
[ ] Use 神州数码 and 浪潮信息 as main cases
[ ] Use 工业富联 as a stable but high-level comparison case
[ ] Use 紫光股份 only as supplementary attention
[ ] Mention limitations clearly

Before resume submission:
[ ] Put 2-3 project bullets into resume
[ ] Prepare 1-minute project explanation
[ ] Prepare 3-minute project explanation
[ ] Upload cleaned project to GitHub
[ ] Add GitHub link to resume if repository is ready
"""

checklist_text = textwrap.dedent(checklist_text).strip()

CHECKLIST_FILE = PORTFOLIO_DIR / "final_project_checklist.txt"
with open(CHECKLIST_FILE, "w", encoding="utf-8") as f:
    f.write(checklist_text)

print("Checklist saved:")
print(CHECKLIST_FILE)

print("\nNote07 completed.")

Checklist saved:
D:\ai_supply_chain_risk\outputs\portfolio_assets\final_project_checklist.txt

Note07 completed.
